# Lab 3: Thompson sampling and linear bandits

The first part implements Thompson sampling for Bernoulli arms and compares it with UCB.
The second part moves to linear bandits: ridge regression with the recursive update of
Tutorial 3, LinUCB and linear Thompson sampling, on a catalog of items.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed". The agent `UCBAgent` of Labs 1 and 2
is now provided by `seqlearn`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import Agent, BernoulliBandit, UCBAgent, plot_mean, pseudo_regret, run_many
from seqlearn import checks
from seqlearn.linear import CatalogBandit, ContextFree, LinearAgent, make_catalog, run_many_linear
from seqlearn.plotting import COLORS, MUTED, beta_pdf, set_style

set_style()

# Part I: Thompson sampling

## 1. The agent

With a uniform prior on each mean, the posterior of arm $a$ after $S_a$ successes and
$F_a$ failures is $\mathrm{Beta}(1 + S_a, 1 + F_a)$ (Tutorial 3, exercise 1). At each
round, Thompson sampling draws $\tilde\theta_a$ from the posterior of each arm and plays
$\arg\max_a \tilde\theta_a$. There are no initialization rounds.

The cell below shows the posterior $\mathrm{Beta}(1 + S, 1 + F)$ of one arm after $n$ pulls
with a proportion $p$ of successes, and ten draws $\tilde\theta$ from it, the values that
Thompson sampling would use. Move the sliders: the posterior concentrates around $p$ as $n$
grows, and so do the draws.

In [ ]:
def show_posterior(n=10, p=0.6):
    successes = round(p * n)
    alpha, beta = 1 + successes, 1 + n - successes
    x = np.linspace(0, 1, 500)
    density = beta_pdf(x, alpha, beta)
    draws = np.random.default_rng(0).beta(alpha, beta, size=10)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(x, density, color=COLORS[0])
    ax.fill_between(x, density, color=COLORS[0], alpha=0.15, linewidth=0)
    ax.plot(draws, np.zeros_like(draws), "o", color=COLORS[1], clip_on=False, label="draws")
    ax.axvline(p, color=MUTED, linestyle="--", linewidth=1)
    ax.set(xlim=(0, 1), ylim=(0, None), yticks=[], xlabel="mean",
           title=f"Beta({alpha}, {beta}): {successes} successes out of {n} pulls")
    ax.legend(loc="upper left")
    plt.show()


try:
    from ipywidgets import FloatSlider, IntSlider, interact
    interact(show_posterior,
             n=IntSlider(value=10, min=0, max=500, step=1, description="pulls n"),
             p=FloatSlider(value=0.6, min=0, max=1, step=0.05, description="successes p"))
except ImportError:
    print("ipywidgets is missing: run `conda install -c conda-forge ipywidgets` in the environment.")
    for n in [0, 10, 100]:
        show_posterior(n, 0.6)

**Question 1.** Complete `ThompsonAgent`. The method `posterior()` returns the two arrays
`(alpha, beta)` of the parameters of the $K$ posteriors, and `act()` draws one sample per
arm and returns the argmax.

*Hint:* the number of successes of arm $a$ is `self.mu_hat[a] * self.counts[a]`, and
`self.rng.beta(alpha, beta)` draws one sample per arm when `alpha` and `beta` are arrays.

In [ ]:
class ThompsonAgent(Agent):
    def posterior(self):
        ...  # TODO

    def act(self):
        ...  # TODO


checks.check_thompson(ThompsonAgent)

The reference instance is the one of Labs 1 and 2: two Bernoulli arms of means $0.6$ and
$0.4$, horizon $T = 5000$.

In [ ]:
MEANS = [0.6, 0.4]
HORIZON = 5000
N_RUNS = 50
t = np.arange(1, HORIZON + 1)


def regret_curves(make_agent, means=MEANS, horizon=HORIZON, n_runs=N_RUNS):
    arms, _ = run_many(lambda rng: BernoulliBandit(means, rng=rng), make_agent, horizon, n_runs)
    return pseudo_regret(arms, BernoulliBandit(means).gaps)


def final_stats(regret):
    final = regret[:, -1]
    return final.mean(), 2 * final.std(ddof=1) / np.sqrt(len(final))

**Question 2.** Run Thompson sampling once on the reference instance and store the
parameters of the posteriors after $10$, $100$ and $1000$ rounds. Plot the two posterior
densities at these three times, on three panels, with `beta_pdf(x, a, b)`, and mark the
true means with vertical lines. Print the number of pulls of each arm after $1000$ rounds.

In [ ]:
env = BernoulliBandit(MEANS, rng=0)
agent = ThompsonAgent(env.n_arms, rng=1)
snapshots = {}
for s in range(1000):
    ...  # TODO

...  # TODO

## 2. Comparison with UCB

**Question 3.** Plot the mean pseudo-regret of UCB1, UCB($0.5$) and Thompson sampling on
the reference instance, with a logarithmic x-axis, together with the Lai-Robbins line
$t \mapsto \frac{\Delta}{\mathrm{kl}(0.4, 0.6)} \log t$, where
$\mathrm{kl}(p, q) = p \log\frac{p}{q} + (1-p)\log\frac{1-p}{1-q}$. Print for each strategy
the mean final regret $\pm 2$ standard errors.

In [ ]:
...  # TODO

**Question 4.** Compare the mean final regret of the three strategies on the instance with
means $(0.6, 0.55)$, then on the instance with five arms of means
$(0.7, 0.6, 0.5, 0.4, 0.2)$.

In [ ]:
...  # TODO

# Part II: Linear bandits

## 3. The framework

In a linear bandit, each arm $a$ has a feature vector $x_a \in \mathbb{R}^d$, known to the
agent, and its mean reward is $\langle \theta^\star, x_a \rangle$, where
$\theta^\star \in \mathbb{R}^d$ is unknown. Rewards are Bernoulli.

An environment has a method `features()`, which returns the array of shape `(n_arms, dim)`
of the feature vectors of the arms, and attributes `theta` (the unknown
$\theta^\star$) and `dim`. A linear agent inherits from `LinearAgent` and implements
`act(features)`, which returns the arm to play, and `update(x, arm, reward)`, where `x` is
the feature vector of the arm that was played.
`run_many_linear(make_env, make_agent, horizon, n_runs)` returns the cumulative
pseudo-regret, of shape `(n_runs, horizon)`; here `make_agent(env, rng)` receives the
environment.

The environment is a catalog of $K = 100$ items described by $d = 5$ features.

In [ ]:
ITEMS, THETA = make_catalog(n_items=100, dim=5, rng=0)
N_RUNS_LINEAR = 20


def make_catalog_env(rng):
    return CatalogBandit(ITEMS, THETA, rng)


env = make_catalog_env(0)
means = env.features() @ env.theta
print("features:", env.features().shape)
print(f"mean rewards: best {means.max():.3f}, median {np.median(means):.3f}, worst {means.min():.3f}")

## 4. Ridge regression

After observing the pairs $(x_1, r_1), \ldots, (x_n, r_n)$, the ridge estimator is
$\hat\theta = V^{-1} b$ with $V = \lambda I + \sum_s x_s x_s^\top$ and $b = \sum_s r_s x_s$
(Tutorial 3, exercise 2). The Sherman-Morrison formula updates the inverse without solving
a linear system:
$$(V + x x^\top)^{-1} = V^{-1} - \frac{V^{-1} x x^\top V^{-1}}{1 + x^\top V^{-1} x}
\enspace.$$

**Question 5.** Complete the class `Ridge`. The method `update(x, r)` updates `V_inv` with
the Sherman-Morrison formula, then `b` and `theta_hat`. The method `width(X)` returns, for
an array `X` of shape `(n_arms, dim)`, the array of the $\sqrt{x^\top V^{-1} x}$ for each
row $x$ of `X`.

*Hint:* with `u = self.V_inv @ x`, the numerator is `np.outer(u, u)`, since $V^{-1}$ is
symmetric. For `width`, `np.sum(X @ self.V_inv * X, axis=1)` computes all the
$x^\top V^{-1} x$ at once.

In [ ]:
class Ridge:
    def __init__(self, dim, lam=1.0):
        self.V_inv = np.eye(dim) / lam
        self.b = np.zeros(dim)
        self.theta_hat = np.zeros(dim)

    def update(self, x, r):
        ...  # TODO

    def width(self, X):
        ...  # TODO


checks.check_ridge(Ridge)

## 5. LinUCB

LinUCB plays the arm maximizing the index
$$U(x) = \langle \hat\theta, x \rangle + \beta \sqrt{x^\top V^{-1} x} \enspace,$$
where $\beta > 0$ is the exploration parameter.

**Question 6.** Complete `LinUCBAgent`: `index(features)` returns the array of the indices
of the arms, `act(features)` the argmax of the index, and `update` passes the observation
to `self.ridge`.

In [ ]:
class LinUCBAgent(LinearAgent):
    def __init__(self, dim, beta=1.0, lam=1.0, rng=None):
        self.beta, self.lam = beta, lam
        super().__init__(dim, rng)

    def reset(self):
        super().reset()
        self.ridge = Ridge(self.dim, self.lam)

    def index(self, features):
        ...  # TODO

    def act(self, features):
        ...  # TODO

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        ...  # TODO


checks.check_linucb(LinUCBAgent)

**Question 7.** On the catalog, compare LinUCB with $\beta = 1$ and UCB1, which ignores the
features and treats the $100$ items as unrelated arms: `ContextFree(UCBAgent(...))` turns a
$K$-armed agent into a linear agent. Plot the mean pseudo-regret over `N_RUNS_LINEAR` runs
of horizon `HORIZON`, and print the mean final regrets.

In [ ]:
...  # TODO

**Question 8.** Plot the mean final regret of LinUCB on the catalog as a function of
$\beta \in \{0.1, 0.3, 1, 2, 4\}$, with $\pm 2$ standard error bars and a logarithmic
x-axis.

In [ ]:
...  # TODO

## 6. Linear Thompson sampling

Linear Thompson sampling draws $\tilde\theta$ from the Gaussian distribution
$\mathcal N(\hat\theta, v^2 V^{-1})$ and plays $\arg\max_a \langle \tilde\theta, x_a \rangle$.

**Question 9.** Complete `LinTSAgent`: `sample()` returns one draw $\tilde\theta$, and
`act(features)` plays the best arm for this draw. Then compare LinUCB with $\beta = 1$ and
linear Thompson sampling with $v = 0.3$ on the catalog (mean final regret $\pm 2$ standard
errors).

*Hint:* if $L$ is the Cholesky factor of $V^{-1}$ (`np.linalg.cholesky`) and
$z \sim \mathcal N(0, I_d)$ (`self.rng.standard_normal(self.dim)`), then
$\hat\theta + v L z$ has the required distribution. `self.rng.multivariate_normal` also
works, but it is about ten times slower.

In [ ]:
class LinTSAgent(LinearAgent):
    def __init__(self, dim, v=0.3, lam=1.0, rng=None):
        self.v, self.lam = v, lam
        super().__init__(dim, rng)

    def reset(self):
        super().reset()
        self.ridge = Ridge(self.dim, self.lam)

    def sample(self):
        ...  # TODO

    def act(self, features):
        ...  # TODO

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        ...  # TODO


checks.check_lints(LinTSAgent)

In [ ]:
...  # TODO

## Going further

**Question 10.** Build catalogs of $K \in \{10, 100, 1000\}$ items with
`make_catalog(K, 5, rng=0)` and plot the mean final regret of UCB1 and of LinUCB with
$\beta = 1$ as a function of $K$, with logarithmic axes.

In [ ]:
...  # TODO

**Question 11.** Plot the mean final regret of linear Thompson sampling on the catalog as a
function of $v \in \{0.03, 0.1, 0.3, 1\}$, as in Question 8.

In [ ]:
...  # TODO